# Capabilities D01–D18 (steady states, simulation, bifurcation, scans)

Prompts exactly as in the paper. Each prompt cell starts a fresh session with `new_chat()`; a multi-turn conversation continues in the cells below it without `new_chat()`. Where the paper's run answered a question from Dynalyst, the reply is in a cell after the prompt, as `reply('yes')`: it sends the reply only if Dynalyst has asked a question in this run, and otherwise does nothing.

To install Dynalyst, see `INSTALL.md` at the root of the repository. For what each notebook holds, see `README.md` in this folder.

## How to use
1. Run the **setup cell**, choose the model and paste its API key.
2. Run the **files cell**: it copies the extra models, the R01 time course and the papers from `model_files/` (next to this notebook) into `data/`.
3. Run any prompt cell. Prompts that use BioModels, PubMed or OmniPath need internet.
4. If a cell hangs, use *Kernel > Interrupt*; after a restart, run the setup and files cells again.

In [ ]:
import os, getpass
# Your API key, asked once and kept only in this session (never saved in the notebook).
# Leave it empty if NVIDIA_API_KEY is already set, or to type the key in the box of the setup cell.
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-..., or Enter to skip): ").strip()

In [ ]:
import sys
from pathlib import Path

# The project folder: the first folder up from here that holds the dynalyst package.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dynalyst" / "__init__.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynalyst import config
from dynalyst.env_check import check_env
from dynalyst.model_picker import pick_model
from dynalyst.notebook import ask, show_answer
from dynalyst.agents import unified_agent
from dynalyst.agents.unified_agent import reset_agents as new_chat   # fresh session per prompt


def ask_auto(query):
    """The prompt as the evaluation scripts sent it: no pause before AUTO-07p."""
    answer = unified_agent.ask(query, approval="auto")
    show_answer(answer)
    return answer

def reply(text="yes"):
    """Answer a question Dynalyst asked (which parameter, wait for a long run, a wider range).
    Sends nothing when no question is open, so a rerun that did not ask is not given a stray "yes"."""
    if not any(a.state["pending_approval"] for a in unified_agent._AGENTS.values()):
        print("No open question from Dynalyst in this run: nothing sent.")
        return None
    return ask(text)

check_env()
print("AUTO-07p:", config.AUTO_DIR)
pick_model();     # choose the model, give its API key if asked

In [ ]:
# Copy the extra model files, data and papers into the project (only files not already there)
import shutil
FILES = Path("model_files") if Path("model_files").exists() else ROOT / "notebooks" / "benchmarks" / "model_files"
DEST = {".xml": config.DATA_DIR / "sbml_models", ".ode": config.DATA_DIR / "sbml_models",
        ".csv": config.DATA_DIR / "timecourses", ".pdf": config.PAPERS_DIR}
for f in sorted(FILES.iterdir()):
    d = DEST.get(f.suffix.lower())
    if d and not (d / f.name).exists():
        d.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, d / f.name)
        print("added", d / f.name)
print("model files ready")

## One example of every capability (main text: Results; S1 File, Section S9)

47 requests, one run each in a clean session.

### D01: Steady states and stability

In [ ]:
new_chat()
ask('Find all steady states of Markevich2004 at MAPKK = 50 nM and MKP3 = 100 nM, and their stability.')

### D02: Jacobian interaction network and feedback loops

In [ ]:
new_chat()
ask('Which feedback loops in the Jacobian interaction network of Markevich2004 could create its switch? Show the network.')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes

### D03: Deterministic simulation

In [ ]:
new_chat()
ask('Simulate Markevich2004 for 2000 time units and describe what happens.')

### D04: Several initial conditions

In [ ]:
new_chat()
ask('At MAPKK = 50, simulate Markevich2004 once starting with all MAPK unphosphorylated (M = 500) and once starting with all MAPK doubly phosphorylated (Mpp = 500), and compare where each ends.')

### D05: Phase plane

In [ ]:
new_chat()
ask('Plot the phase plane of Mp against Mpp for Markevich2004.')

### D06: Rates of change

In [ ]:
new_chat()
ask('Plot the net rate of change of each species of Markevich2004 over the first 500 time units.')

### D07: Custom plot

In [ ]:
new_chat()
ask('Plot M, Mp and Mpp of Markevich2004 from t = 0 to 1000 with a logarithmic y axis.')

### D08: Stochastic simulation (Gillespie SSA)

In [ ]:
new_chat()
ask('Run 5 Gillespie SSA runs of Markevich2004 at MAPKK = 48 nM with system size 0.1 up to t = 200000 and say whether the runs switch between states.')

### D09: Stochastic against deterministic

In [ ]:
new_chat()
ask('Compare stochastic and deterministic simulations of Markevich2004 at system size 1.')

### D10: Propensity table

In [ ]:
new_chat()
ask('Show the reaction channels of Markevich2004 and their propensities at the initial state.')

### D11: Bifurcation diagram (AUTO-07P)

In [ ]:
new_chat()
ask('Run AUTO-07P on Markevich2004 with MAPKK as the continuation parameter over [0, 150], plot Mpp, and report all fold points.')

### D12: Choosing the continuation parameter (with approval)

In [ ]:
new_chat()
ask('Compute a bifurcation diagram for Markevich2004.')

In [ ]:
# The paper's replies: parameter: "yes".
# reply() sends only if Dynalyst asked; with two questions, run the line that answers the one asked:
reply('yes')

### D13: Candidate parameters without AUTO-07P

In [ ]:
new_chat()
ask('Which parameters of Markevich2004 could give bistability or oscillations? Do not run AUTO-07P.')

### D14: Plain parameter sweep

In [ ]:
new_chat()
ask('Sweep MKP3 from 50 to 150 nM in Markevich2004 and show how the steady-state Mpp changes, without continuation.')

### D15: Latin hypercube scan, a diagram per sample, classified

In [ ]:
new_chat()
ask('Sample 20 parameter sets of Markevich2004, varying k1cat, k2cat, k3cat and k4cat by plus or minus 50 % around their values, run a bifurcation diagram in MAPKK for each, and classify them.')

### D16: Latin hypercube survey of bifurcation kinds

In [ ]:
new_chat()
ask('Survey Markevich2004: over 20 samples with all rate constants varied by plus or minus 50 %, how often does each kind of bifurcation occur?')

In [ ]:
# The paper's replies: parameter: "yes".
# reply() sends only if Dynalyst asked; with two questions, run the line that answers the one asked:
reply('yes')

### D17: Textbook bifurcation example

In [ ]:
new_chat()
ask('Show me a textbook example of a Hopf bifurcation with its bifurcation diagram.')

In [ ]:
# The paper's replies: cycles: "no".
# reply() sends only if Dynalyst asked; with two questions, run the line that answers the one asked:
# reply('no')

### D18: Hopf point and limit cycles

In [ ]:
new_chat()
ask('Find the Hopf bifurcation of BIOMD0000000010 in Ki and follow the limit cycles from it; report the period at the Hopf point and how it changes with Ki.')

In [ ]:
# The paper's replies: parameter: "yes"; cycles: "yes".
# reply() sends only if Dynalyst asked; with two questions, run the line that answers the one asked:
reply('yes')
# reply('yes')

### D04b: Several initial conditions, long enough to settle

In [ ]:
new_chat()
ask('At MAPKK = 50, simulate Markevich2004 for 10000 time units once from M = 500, Mp = 0, Mpp = 0 and once from M = 0, Mp = 0, Mpp = 500, and compare the steady states they reach.')

### D10b: Propensity table, named

In [ ]:
new_chat()
ask('Use the propensity table tool to list the reaction channels of Markevich2004 and their propensities at the initial state, at system size 1.')

### D13b: Eigenvalue screen of candidate parameters, named

In [ ]:
new_chat()
ask('Rank the parameters of Markevich2004 that could give a fold or a Hopf point with the eigenvalue screen (propose bifurcation candidates); do not run AUTO-07P.')

In [ ]:
reply('yes')   # Dynalyst asked (budget_wait); reply yes